In [ ]:
# Baseline CPU FP32 Inference for PYNQ

import numpy as np
import json
import time

BATCH = 1 # batch size

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

# Dequantize weights to FP32 once
W = {
    "fc1": w["fc1"].astype(np.float32) * ws["fc1"], # (128, 784)
    "fc2": w["fc2"].astype(np.float32) * ws["fc2"], # (64, 128)
    "fc3": w["fc3"].astype(np.float32) * ws["fc3"], # (10, 64)
}


def run_cpu_gemm(x_fp32, w_fp32, M, K, N):
    return x_fp32 @ w_fp32.T

def bias_relu(y_fp32, bias):
    return np.maximum(y_fp32 + bias, 0.0).astype(np.float32)

def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(logits)
    return (exp_x / np.sum(exp_x, axis=1, keepdims=True)).astype(np.float32)


def predict_batch(images_fp32):
    M = images_fp32.shape[0]

    # FC1: A=(M,784), B=(784,128) -> C=(M,128)
    x_fp32 = images_fp32.reshape(M, -1)
    fc1 = run_cpu_gemm(x_fp32, W["fc1"], M=M, K=784, N=128)
    x_fp32 = bias_relu(fc1, b["fc1"])

    # FC2: A=(M,128), B=(128,64) -> C=(M,64)
    fc2 = run_cpu_gemm(x_fp32, W["fc2"], M=M, K=128, N=64)
    x_fp32 = bias_relu(fc2, b["fc2"])

    # FC3: A=(M,64), B=(64,10) -> C=(M,10)
    fc3 = run_cpu_gemm(x_fp32, W["fc3"], M=M, K=64, N=10)
    logits = fc3 + b["fc3"]

    probs = softmax(logits)
    return np.argmax(probs, axis=1), logits, probs


# Sample predictions (first 5 images as a batch)
print(f"===== CPU FP32 Predictions (batch={BATCH}) =====")
preds, logits, probs = predict_batch(images[:BATCH])
for idx in range(5):
    print(f"Sample {idx}: label={labels[idx]}, pred={preds[idx]}")
    print(f"  logits = {np.round(logits[idx], 4)}")
    print(f"  probs  = {np.round(probs[idx],  4)}")

# Accuracy on first 1000 samples
print("\n===== CPU FP32 Accuracy =====")
correct = 0
for i in range(0, 1000, BATCH):
    end = min(i + BATCH, 1000)
    batch_imgs = images[i:end]
    batch_preds, _, _ = predict_batch(batch_imgs)
    correct += np.sum(batch_preds == labels[i:end])
print(f"Accuracy on first 1000 samples: {correct / 1000:.4f}")

# Latency benchmark
print("\n===== CPU FP32 Timing =====")
n_batches = 100 // BATCH
start = time.perf_counter()
for i in range(n_batches):
    predict_batch(images[i*BATCH:(i+1)*BATCH])
total_samples = n_batches * BATCH
avg_ms = (time.perf_counter() - start) * 1000.0 / total_samples
print(f"Batch size:                {BATCH}")
print(f"Average CPU FP32 latency:  {avg_ms:.4f} ms/sample")


In [ ]:
# Baseline CPU INT8 Inference for PYNQ

import numpy as np
import json
import time

BATCH = 1 # batch size

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")


def run_cpu_gemm(x_int8, w_int8, M, K, N):
    return x_int8.astype(np.int32) @ w_int8.T.astype(np.int32)

def quantize(x_fp32, scale):
    return np.clip(np.round(x_fp32 / scale), -128, 127).astype(np.int8)

def dequantize_bias_relu(y_int32, x_scale, w_scale, bias):
    y = y_int32.astype(np.float32) * (x_scale * w_scale) + bias
    return np.maximum(y, 0.0).astype(np.float32)

def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(logits)
    return (exp_x / np.sum(exp_x, axis=1, keepdims=True)).astype(np.float32)


def predict_batch(images_fp32):
    M = images_fp32.shape[0]

    # FC1: A=(M,784), B=(784,128) -> C=(M,128)
    x_int8 = quantize(images_fp32.reshape(M, -1), act_scales["input"])
    fc1 = run_cpu_gemm(x_int8, w["fc1"], M=M, K=784, N=128)
    x_int8 = quantize(dequantize_bias_relu(fc1, act_scales["input"], ws["fc1"], b["fc1"]),
                      act_scales["relu1_out"])

    # FC2: A=(M,128), B=(128,64) -> C=(M,64)
    fc2 = run_cpu_gemm(x_int8, w["fc2"], M=M, K=128, N=64)
    x_int8 = quantize(dequantize_bias_relu(fc2, act_scales["relu1_out"], ws["fc2"], b["fc2"]),
                      act_scales["relu2_out"])

    # FC3: A=(M,64), B=(64,10) -> C=(M,10)
    fc3 = run_cpu_gemm(x_int8, w["fc3"], M=M, K=64, N=10)
    logits = fc3.astype(np.float32) * (act_scales["relu2_out"] * ws["fc3"]) + b["fc3"]

    probs = softmax(logits)
    return np.argmax(probs, axis=1), logits, probs


# Sample predictions (first 5 images as a batch)
print(f"===== CPU Batched Predictions (batch={BATCH}) =====")
preds, logits, probs = predict_batch(images[:BATCH])
for idx in range(5):
    print(f"Sample {idx}: label={labels[idx]}, pred={preds[idx]}")
    print(f"  logits = {np.round(logits[idx], 4)}")
    print(f"  probs  = {np.round(probs[idx],  4)}")

# Accuracy on first 1000 samples
print("\n===== CPU Accuracy =====")
correct = 0
for i in range(0, 1000, BATCH):
    end = min(i + BATCH, 1000)
    batch_imgs = images[i:end]
    batch_preds, _, _ = predict_batch(batch_imgs)
    correct += np.sum(batch_preds == labels[i:end])
print(f"Accuracy on first 1000 samples: {correct / 1000:.4f}")

# Latency benchmark
print("\n===== CPU Timing =====")
n_batches = 100 // BATCH
start = time.perf_counter()
for i in range(n_batches):
    predict_batch(images[i*BATCH:(i+1)*BATCH])
total_samples = n_batches * BATCH
avg_ms = (time.perf_counter() - start) * 1000.0 / total_samples
print(f"Batch size:                {BATCH}")
print(f"Average CPU latency:       {avg_ms:.4f} ms/sample")


In [ ]:
# Multi-GEMM Accelerated Inference

import numpy as np
import json
import time
from pynq import Overlay, allocate

BATCH = 1 # batch size

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

ol = Overlay("block_design.bit")
ip = ol.multiply_0

# Pre-load weight buffers, so they're never copied again during inference
layers = [
    {"name": "fc1", "K": 784, "N": 128},
    {"name": "fc2", "K": 128, "N":  64},
    {"name": "fc3", "K":  64, "N":  10},
]

for layer in layers:
    K, N = layer["K"], layer["N"]
    buf = allocate(shape=(K * N,), dtype=np.int8)
    buf[:] = w[layer["name"]].T.flatten() # transpose to (K, N) row-major
    buf.sync_to_device() # sync once, never again
    layer["B_buf"] = buf

# A and C buffers sized for FC1 at full batch size
A_buf = allocate(shape=(BATCH * 784,), dtype=np.int8)
C_buf = allocate(shape=(BATCH * 128,), dtype=np.int32)


def run_fpga_gemm(x_int8, B_buf, M, K, N):
    # Run one batched GEMM
    # A=(M,K), B=(K,N), C=(M,N)
    A_buf[:M * K] = x_int8.reshape(-1)
    A_buf.sync_to_device()

    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x1c, B_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (B_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x28, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x2c, (C_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x34, M)
    ip.write(0x3c, K)
    ip.write(0x44, N)

    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass

    C_buf.sync_from_device()
    return np.array(C_buf[:M * N]).reshape(M, N)


def quantize(x_fp32, scale):
    return np.clip(np.round(x_fp32 / scale), -128, 127).astype(np.int8)

def dequantize_bias_relu(y_int32, x_scale, w_scale, bias):
    y = y_int32.astype(np.float32) * (x_scale * w_scale) + bias
    return np.maximum(y, 0.0).astype(np.float32)

def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(logits)
    return (exp_x / np.sum(exp_x, axis=1, keepdims=True)).astype(np.float32)


def predict_batch(images_fp32):
    # Run inference on a batch of images
    # Returns array of BATCH predictions
    M = images_fp32.shape[0]

    # FC1: A=(M,784), B=(784,128) -> C=(M,128)
    x_int8 = quantize(images_fp32.reshape(M, -1), act_scales["input"])
    fc1 = run_fpga_gemm(x_int8, layers[0]["B_buf"], M=M, K=784, N=128)
    x_int8 = quantize(dequantize_bias_relu(fc1, act_scales["input"], ws["fc1"], b["fc1"]),
                      act_scales["relu1_out"])

    # FC2: A=(M,128), B=(128,64) -> C=(M,64)
    fc2 = run_fpga_gemm(x_int8, layers[1]["B_buf"], M=M, K=128, N=64)
    x_int8 = quantize(dequantize_bias_relu(fc2, act_scales["relu1_out"], ws["fc2"], b["fc2"]),
                      act_scales["relu2_out"])

    # FC3: A=(M,64), B=(64,10) -> C=(M,10)
    fc3 = run_fpga_gemm(x_int8, layers[2]["B_buf"], M=M, K=64, N=10)
    logits = fc3.astype(np.float32) * (act_scales["relu2_out"] * ws["fc3"]) + b["fc3"]

    probs = softmax(logits)
    return np.argmax(probs, axis=1), logits, probs


# Sample predictions (first 5 images as a batch)
print(f"===== FPGA Batched Predictions (batch={BATCH}) =====")
preds, logits, probs = predict_batch(images[:BATCH])
for idx in range(5):
    print(f"Sample {idx}: label={labels[idx]}, pred={preds[idx]}")
    print(f"  logits = {np.round(logits[idx], 4)}")
    print(f"  probs  = {np.round(probs[idx],  4)}")

# Accuracy on first 1000 samples
print("\n===== FPGA Accuracy =====")
correct = 0
for i in range(0, 1000, BATCH):
    end = min(i + BATCH, 1000)
    batch_imgs = images[i:end]
    batch_preds, _, _ = predict_batch(batch_imgs)
    correct += np.sum(batch_preds == labels[i:end])
print(f"Accuracy on first 1000 samples: {correct / 1000:.4f}")

# Latency benchmark
print("\n===== FPGA Timing =====")
n_batches = 100 // BATCH  # run enough batches to cover 100 samples
start = time.perf_counter()
for i in range(n_batches):
    predict_batch(images[i*BATCH:(i+1)*BATCH])
total_samples = n_batches * BATCH
avg_ms = (time.perf_counter() - start) * 1000.0 / total_samples
print(f"Batch size:                {BATCH}")
print(f"Average FPGA latency:      {avg_ms:.4f} ms/sample")

# Cleanup
A_buf.close()
C_buf.close()
for layer in layers:
    layer["B_buf"].close()


In [ ]:
# Monolithic Only Accelerated Inference

import numpy as np
import json
import time
from pynq import Overlay, allocate

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
# b = {
#     "fc1": np.load("artifacts/fc1_bias_int32.npy").astype(np.int32),
#     "fc2": np.load("artifacts/fc2_bias_int32.npy").astype(np.int32),
#     "fc3": np.load("artifacts/fc3_bias_int32.npy").astype(np.int32),
# }

with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

ol = Overlay("monolithic_only.bit")   
ip = ol.mlp_0

A_buf  = allocate(shape=(784,), dtype=np.int8)
W1_buf = allocate(shape=(784 * 128,), dtype=np.int8)
b1_buf = allocate(shape=(128,), dtype=np.int32)
W2_buf = allocate(shape=(128 * 64,), dtype=np.int8)
b2_buf = allocate(shape=(64,), dtype=np.int32)
W3_buf = allocate(shape=(64 * 10,), dtype=np.int8)
b3_buf = allocate(shape=(10,), dtype=np.int32)
C_buf  = allocate(shape=(10,), dtype=np.int32)


W1_buf[:] = w["fc1"].T.reshape(-1)
W2_buf[:] = w["fc2"].T.reshape(-1)
W3_buf[:] = w["fc3"].T.reshape(-1)

b1_buf[:] = b["fc1"]
b2_buf[:] = b["fc2"]
b3_buf[:] = b["fc3"]

W1_buf.sync_to_device()
b1_buf.sync_to_device()
W2_buf.sync_to_device()
b2_buf.sync_to_device()
W3_buf.sync_to_device()
b3_buf.sync_to_device()

def quantize_input(x_fp32):
    x_int8 = np.clip(
        np.round(x_fp32.reshape(-1) / act_scales["input"]),
        -128, 127
    ).astype(np.int8)
    return x_int8

def run_mlp_one_image(x_fp32):
    x_int8 = quantize_input(x_fp32)

    A_buf[:] = x_int8
    A_buf.sync_to_device()


    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x1c, W1_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (W1_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x28, b1_buf.device_address & 0xFFFFFFFF)
    ip.write(0x2c, (b1_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x34, W2_buf.device_address & 0xFFFFFFFF)
    ip.write(0x38, (W2_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x40, b2_buf.device_address & 0xFFFFFFFF)
    ip.write(0x44, (b2_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x4c, W3_buf.device_address & 0xFFFFFFFF)
    ip.write(0x50, (W3_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x58, b3_buf.device_address & 0xFFFFFFFF)
    ip.write(0x5c, (b3_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x64, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x68, (C_buf.device_address >> 32) & 0xFFFFFFFF)

    # start
    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass

    C_buf.sync_from_device()
    logits = np.array(C_buf, dtype=np.int32)
    pred = int(np.argmax(logits))
    return pred, logits


print("===== Monolithic MLP FPGA Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)


correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])

print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")


n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average FPGA latency: {avg_ms:.4f} ms/sample")


A_buf.close()
W1_buf.close()
b1_buf.close()
W2_buf.close()
b2_buf.close()
W3_buf.close()
b3_buf.close()
C_buf.close()

In [ ]:
# Monolithic Hardcoded Weights Accelerated Inference

import numpy as np
import json
import time
from pynq import Overlay, allocate

# Load all artifacts
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")


ol = Overlay("monolithic_hardcode.bit")   
ip = ol.mlp_0                      


A_buf = allocate(shape=(784,), dtype=np.int8)
C_buf = allocate(shape=(10,), dtype=np.int32)

def quantize_input(x_fp32):
    x_int8 = np.clip(
        np.round(x_fp32.reshape(-1) / act_scales["input"]),
        -128, 127
    ).astype(np.int8)
    return x_int8

def run_mlp_one_image(x_fp32):
    x_int8 = quantize_input(x_fp32)

    A_buf[:] = x_int8
    A_buf.sync_to_device()


    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x1c, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (C_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass

    C_buf.sync_from_device()
    logits = np.array(C_buf, dtype=np.int32)
    pred = int(np.argmax(logits))
    return pred, logits


print("===== Hardcoded Monolithic FPGA Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)


correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])

print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")


n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average FPGA latency: {avg_ms:.4f} ms/sample")


A_buf.close()
C_buf.close()